# CUDA Device Setup for Faster Training

This notebook demonstrates how to set up and use CUDA (GPU) for faster model training in PyTorch. Using GPU can speed up training by 10-100x compared to CPU, especially for deep learning models.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Check if CUDA is available
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("PyTorch Version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU Device Count:", torch.cuda.device_count())
    print("Current GPU Device:", torch.cuda.current_device())
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("GPU Memory Total:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")
else:
    print("CUDA is not available. Please check your GPU drivers and PyTorch installation.")

CUDA Available: True
CUDA Version: 11.8
PyTorch Version: 2.7.1+cu118
GPU Device Count: 1
Current GPU Device: 0
GPU Device Name: NVIDIA GeForce RTX 3070 Laptop GPU
GPU Memory Total: 7.99951171875 GB


In [2]:
# Set up device (CUDA if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# If using GPU, print additional info
if device.type == 'cuda':
    print(f"GPU Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Using device: cuda
GPU Memory Available: 8.00 GB
GPU Memory Allocated: 0.00 GB
GPU Memory Cached: 0.00 GB


# ⚠️ CUDA Not Available - Installation Required

Since CUDA is not available, you need to install the CUDA-enabled version of PyTorch. Follow these steps:

## Step 1: Check if you have an NVIDIA GPU
1. Open Device Manager (Win + X, then M)
2. Look under "Display adapters" for NVIDIA GPU
3. If you don't see NVIDIA GPU, you cannot use CUDA

## Step 2: Install NVIDIA GPU Drivers
- Download and install the latest drivers from [NVIDIA's website](https://www.nvidia.com/drivers/)
- Restart your computer after installation

## Step 3: Install CUDA-enabled PyTorch
Run ONE of these commands in your terminal/command prompt:

### For CUDA 11.8 (Recommended):
```bash
pip uninstall torch torchvision torchaudio
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
```

### For CUDA 12.1:
```bash
pip uninstall torch torchvision torchaudio
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

### For the latest CUDA version:
```bash
pip uninstall torch torchvision torchaudio
pip install torch torchvision torchaudio
```

## Step 4: Restart VS Code
After installation, restart VS Code and re-run the CUDA check cell above.

In [3]:
# After installing CUDA-enabled PyTorch, restart the kernel and run this cell
# You can restart kernel with: Ctrl+Shift+P -> "Python: Restart Kernel"

import torch
print("=== CUDA Status After Installation ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print("✅ CUDA is ready for use!")
else:
    print("❌ CUDA is still not available")
    print("Try restarting VS Code completely")

=== CUDA Status After Installation ===
CUDA Available: True
PyTorch Version: 2.7.1+cu118
CUDA Version: 11.8
GPU Device: NVIDIA GeForce RTX 3070 Laptop GPU
GPU Memory: 8.00 GB
✅ CUDA is ready for use!


In [4]:
# Example: How to move your model and data to GPU
# This is how you should modify your existing training code

# 1. Move your model to GPU
# model = model.to(device)  # Move model to GPU

# 2. Move your data to GPU (in your training loop)
# inputs, labels = inputs.to(device), labels.to(device)

# Example with a simple CNN model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Create model and move to GPU
model = SimpleCNN(num_classes=10)
model = model.to(device)  # This moves the model to GPU
print(f"Model is on device: {next(model.parameters()).device}")

# Example data (batch_size=32, channels=3, height=32, width=32)
sample_input = torch.randn(32, 3, 32, 32).to(device)  # Move data to GPU
sample_labels = torch.randint(0, 10, (32,)).to(device)  # Move labels to GPU

print(f"Sample input is on device: {sample_input.device}")
print(f"Sample labels are on device: {sample_labels.device}")

Model is on device: cuda:0
Sample input is on device: cuda:0
Sample labels are on device: cuda:0


In [5]:
# Example training loop with GPU
def train_step_example():
    # Set model to training mode
    model.train()
    
    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Example training step
    optimizer.zero_grad()
    
    # Forward pass (data is already on GPU)
    outputs = model(sample_input)
    loss = criterion(outputs, sample_labels)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    print(f"Training loss: {loss.item():.4f}")
    print(f"Loss tensor is on device: {loss.device}")
    
    return loss.item()

# Run example training step
loss_value = train_step_example()

Training loss: 2.3383
Loss tensor is on device: cuda:0
